# From pixels to probabilities: image segmentation with graphs

This project connects images, graphs, and linear systems to show how mathematical ideas become a working application. You will explore how modeling choices affect the result and how data representation and implementation affect speed and memory use.

**CS 301 · Unit II mini-project**

[Open this assignment in Google Colab](https://colab.research.google.com/github/ikoutis/cs301-labs/blob/main/unit-2/assignments/image-segmentation.ipynb)

How can a few clicks separate an object from its background? You will turn a color image into a weighted graph, prescribe labels at a few pixels, and solve for the remaining pixels. The same problem will let you compare Python and compiled loops, dense and sparse storage, and dense and sparse linear solvers.

**By the end, you should be able to:**

- Map between image coordinates and graph vertices; build 4- and 8-neighbor affinities.
- Construct a graph Laplacian and explain its entries.
- Derive a linear system from fixed seed labels and a weighted-average rule.
- Compare execution times and memory while checking that answers agree.
- Explain how seeds and affinity weights change a segmentation.

**Working time:** allow several work sessions. Use a CPU runtime; no GPU is needed.

**Name:** _Your name_  
**Image and intended foreground:** _Complete after choosing an image._

## How to work through this notebook

Save your own copy in Colab. Complete the four code exercises, marked **Your implementation**, and each written response. Supplied helpers handle loading, clicking, plotting, and timing; read their contracts before using them. Start with the saved demonstration seeds, then choose your own seeds and investigate a second image.

The starter contains `...` in the implementation exercises. Replace these before running the cells that depend on them. Checks explain common errors; they do not replace your explanations.

**Your submission is one completed `.ipynb`:** all code cells run in order, no errors, outputs and static figures visible, saved seeds for both images, experiment notes, and the final reflection. Restart the runtime and run everything before downloading. The optional bonus has a switch so the main notebook runs without completing it. Submit the notebook, not just its sharing link.

You may use AI to help generate or modify the supplied visualization and interaction helpers. Record the prompt you used, what you changed, and how you verified it. Follow the course's AI rules for all other work, and explain the mathematical code in your own words.


## 1. Load a color image

Run the setup once. Colab normally includes NumPy, SciPy, Matplotlib, Pillow, pandas, and Numba. The cell installs Numba or the small thread-control helper if either is missing. The version table records the environment behind your measurements. Timing results are specific to your runtime.


In [ ]:
import importlib.util
import sys
import subprocess
for package in ("numba", "threadpoolctl"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import io
import json
import platform
import time
from pathlib import Path
from urllib.request import urlopen
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy import sparse
from scipy.sparse.linalg import spsolve, splu
import numba
from numba import njit
from PIL import Image
from IPython.display import display
from threadpoolctl import threadpool_limits, threadpool_info

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"font.size": 10, "figure.dpi": 110})
display(pd.DataFrame({"package": ["Python", "NumPy", "SciPy", "Numba", "Matplotlib"],
                      "version": [platform.python_version(), np.__version__, scipy.__version__,
                                  numba.__version__, matplotlib.__version__]}))
print("CPU:", platform.processor() or platform.machine())
print("Dense solver benchmarks below use one BLAS thread.")


In [ ]:
BASE = "https://ikoutis.github.io/cs301-labs/unit-2/images/"
catalog = pd.read_csv(BASE + "images.csv").set_index("number")

def load_image(number, max_side=128):
    """Return (RGB float64 array in [0,1], catalog row); preserve aspect ratio."""
    if not 8 <= max_side <= 256:
        raise ValueError("Use a longest side between 8 and 256 for this assignment.")
    info = catalog.loc[number]
    with urlopen(BASE + info["file"], timeout=30) as response:
        original = Image.open(io.BytesIO(response.read())).convert("RGB")
    original.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
    return np.ascontiguousarray(np.asarray(original, dtype=np.float64) / 255.0), info

display(catalog[["title", "creator", "height", "width"]])


The course repository contains 30 images. Its [image credits and usage terms](https://ikoutis.github.io/cs301-labs/unit-2/images/SOURCES.md) accompany the collection. Keep the creator and source in your submission. We start with image 2, Earth, and try to separate its disk from the dark background. Later, try a less distinct boundary on a second image.

**Pixels are data.** An image with shape `(H, W, 3)` has `H*W` pixels and three color values per pixel. Convert unsigned integer colors to floating point **before** subtracting them, or subtraction can wrap around. We use RGB distance as a simple model of similarity; it does not capture all visual or semantic differences.


In [ ]:
IMAGE_NUMBER = 2
MAX_SIDE = 128
image, image_info = load_image(IMAGE_NUMBER, MAX_SIDE)
H, W = image.shape[:2]
print(f"{image_info['title']} — {image_info['creator']}")
print("Source:", image_info["source"])
print("Working shape:", image.shape, "| pixels:", H * W, "| dtype:", image.dtype)
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image)
ax.set(xlabel="column c", ylabel="row r (increases downward)", title="Choose an object and its background")
plt.show()


## 2. Represent the pixels as vertices

Number pixels row by row: $i=rW+c$. Recover the coordinates with $r=i\mathbin{//}W$ and $c=i\bmod W$. This is the default order of NumPy's `reshape`.

A **4-neighbor** pixel connects to its left, right, upper, and lower neighbors when they exist. An **8-neighbor** pixel also connects to the four diagonals. No edge crosses from the end of a row to the beginning of the next row.


In [ ]:
vertex_numbers = np.arange(12).reshape(3, 4)
print("Vertex numbers in a 3-by-4 image:\n", vertex_numbers)
r, c = 1, 2
i = r * 4 + c
print(f"(row={r}, column={c}) -> vertex {i} -> coordinates {divmod(i, 4)}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, connectivity in zip(axes, (4, 8)):
    offsets = [(0, 1), (1, 0)] + ([(1, 1), (1, -1)] if connectivity == 8 else [])
    for r in range(3):
        for c in range(4):
            for dr, dc in offsets:
                rr, cc = r + dr, c + dc
                if 0 <= rr < 3 and 0 <= cc < 4:
                    ax.plot([c, cc], [r, rr], color="#aab6c5", lw=1.3, zorder=0)
            ax.scatter(c, r, s=320, color="#2878b5")
            ax.text(c, r, str(r * 4 + c), color="white", ha="center", va="center")
    ax.set_title(f"{connectivity}-neighbor grid")
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
plt.tight_layout(); plt.show()


**Predict.** In a 3-by-4 image, list the 4-neighbors and 8-neighbors of vertex 6. How many undirected edges does each grid contain? Count each pair only once. Explain why connecting every index `i` to `i+1` would create an incorrect edge.


**Your response:** _Replace this text._


### Your implementation 1: affinity edges

For neighboring pixels, use
$$
w_{ij}=\max\{\exp(-\beta\|I_i-I_j\|_2^2),\varepsilon\},\qquad \varepsilon=10^{-6}.
$$
Non-neighbors have weight zero. Similar colors produce strong edges. We give diagonal edges the same formula as horizontal and vertical edges; there is no extra distance factor in this assignment.

The small positive floor prevents numerical underflow from disconnecting the grid. It also changes extremely weak edges, so it is part of the model. Very large $\beta$ values can still make the system poorly conditioned. Start with `beta=80` and keep your experiments between 0 and 300.

Complete the loop below. Each undirected pair is visited once, then stored in **both directions** so that the adjacency matrix is symmetric. The output is three one-dimensional arrays: `rows[t]`, `cols[t]`, and `weights[t]` describe one matrix entry. Preallocated NumPy arrays keep the numerical loop suitable for Numba.

**Predict before running:** what weights result when `beta=0`? What happens to the affinity between different colors when beta increases?


In [ ]:
def affinity_edges(image, beta=80.0, connectivity=4, epsilon=1e-6):
    h, w, channels = image.shape
    assert channels == 3 and connectivity in (4, 8)
    assert beta >= 0 and 0 < epsilon <= 1
    edge_count = h * (w - 1) + (h - 1) * w
    if connectivity == 8:
        edge_count += 2 * (h - 1) * (w - 1)
    rows = np.empty(2 * edge_count, dtype=np.int64)
    cols = np.empty(2 * edge_count, dtype=np.int64)
    weights = np.empty(2 * edge_count, dtype=np.float64)
    offsets = ((0, 1), (1, 0), (1, 1), (1, -1))
    count = 2 if connectivity == 4 else 4
    t = 0
    for r in range(h):
        for c in range(w):
            i = r * w + c
            for direction in range(count):
                dr, dc = offsets[direction]
                rr, cc = r + dr, c + dc
                if 0 <= rr < h and 0 <= cc < w:
                    j = ...  # vertex number of (rr, cc)
                    squared_distance = 0.0
                    for channel in range(3):
                        difference = image[r, c, channel] - image[rr, cc, channel]
                        squared_distance += ...
                    weight = ...  # exponential affinity, with the positive floor
                    rows[t], cols[t], weights[t] = i, j, weight
                    rows[t + 1], cols[t + 1], weights[t + 1] = j, i, weight
                    t += 2
    return rows, cols, weights


In [ ]:
flat = np.zeros((2, 3, 3), dtype=np.float64)
for connectivity, expected_edges, expected_degree in [(4, 7, [2, 3, 2, 2, 3, 2]),
                                                       (8, 11, [3, 5, 3, 3, 5, 3])]:
    rows, cols, weights = affinity_edges(flat, connectivity=connectivity)
    assert len(rows) == 2 * expected_edges, "Count each pair once, then store both directions."
    assert len(set(zip(rows, cols))) == len(rows), "Duplicate edges were generated."
    test_W = sparse.coo_matrix((weights, (rows, cols)), shape=(6, 6)).toarray()
    assert np.allclose(test_W, test_W.T) and np.allclose(test_W.diagonal(), 0)
    assert np.allclose(test_W.sum(axis=1), expected_degree)
    assert test_W[2, 3] == 0, "An edge wrapped across a row boundary."

red_blue = np.array([[[1., 0., 0.], [0., 0., 1.]]])
assert np.allclose(affinity_edges(red_blue, beta=2)[2], np.exp(-4))
assert np.allclose(affinity_edges(red_blue, beta=0)[2], 1)
assert np.allclose(affinity_edges(red_blue, beta=300)[2], 1e-6)
print("Affinity checks passed: boundaries, symmetry, diagonal neighbors, RGB distances, and floor.")


Explain what the three edge arrays represent. Answer the beta predictions above, and explain why we store each undirected edge twice. For a general `H`-by-`W` image, derive the number of undirected edges for each connectivity.


**Your response:** _Replace this text._


## 3. Compile the same loop with Numba

Python executes ordinary loops through its interpreter. Numba can compile supported numerical Python into machine code. `njit` means **no-Python just-in-time compilation**, and is shorthand for `jit(nopython=True)`. The first call compiles a specialization for the argument types. Later calls with those types reuse it. Unsupported operations cause a compilation error; adding the decorator does not make every Python library compilable.

> **NJIT meets `njit`.** You're already enrolled in one; now your Python loops can join the other.

`njit(affinity_edges)` is equivalent to putting `@njit` immediately above the function definition, and lets us keep both versions. Compile the numerical loop; create the SciPy sparse object afterward. Check correctness before comparing timings.


In [ ]:
def median_seconds(function, repeats=3):
    """Time fresh calls; return the median seconds and the final result."""
    samples = []
    result = None
    for _ in range(repeats):
        start = time.perf_counter()
        result = function()
        samples.append(time.perf_counter() - start)
    return float(np.median(samples)), result


In [ ]:
fast_edges = njit(affinity_edges)
timing_image, _ = load_image(IMAGE_NUMBER, 256)
start = time.perf_counter()
compiled_result = fast_edges(timing_image, 80.0, 8, 1e-6)
first_call_seconds = time.perf_counter() - start
python_seconds, python_result = median_seconds(lambda: affinity_edges(timing_image, 80.0, 8, 1e-6))
numba_seconds, compiled_result = median_seconds(lambda: fast_edges(timing_image, 80.0, 8, 1e-6))
assert all(np.allclose(a, b) for a, b in zip(python_result, compiled_result))
display(pd.DataFrame({"measurement": ["Python median", "Numba first call (compile + execute)",
                                      "Numba warm median"],
                      "seconds": [python_seconds, first_call_seconds, numba_seconds]}))
print(f"Warm speedup: {python_seconds / numba_seconds:.1f}x; image shape: {timing_image.shape}")
del python_result, compiled_result


Report the first-call time and the warm speedup. Why would timing just one Numba call give a misleading comparison? Did compilation change the algorithm's number of edges or its asymptotic work? Explain which part became faster. If you rerun the function-definition cell, rerun the compilation cell too.


**Your response:** _Replace this text._


## 4. Build the Laplacian in two storage formats

Write the affinity matrix as $W$, the weighted degree of vertex $i$ as $d_i=\sum_jW_{ij}$, and $D=\operatorname{diag}(d)$. The **unnormalized graph Laplacian** is
$$
L=D-W.
$$
Its diagonal contains weighted degrees, its off-diagonal entries are negative affinities, and every row sums to zero. $L$ is singular: $L\mathbf{1}=0$. We will fix seed values before solving.

**Dense storage** holds every entry, including zeros. **COO storage** starts with the three edge arrays. **CSR storage** groups nonzeros by row and is convenient for slicing and multiplication. Construct the large sparse matrices directly from the edges; do not allocate a dense matrix first.

### Your implementation 2

Complete the two Laplacian functions. The dense version is restricted to at most 1,600 vertices, and the solver benchmarks will use at most 1,024. This is a teaching limit, not a prediction of every computer's capacity.


In [ ]:
def dense_laplacian(n, rows, cols, weights):
    if n > 1600:
        raise ValueError("Dense safety limit: use at most 1,600 pixels.")
    W_dense = np.zeros((n, n), dtype=np.float64)
    np.add.at(W_dense, (rows, cols), weights)
    degree = ...
    L_dense = ...
    return W_dense, L_dense

def sparse_laplacian(n, rows, cols, weights):
    W_sparse = sparse.coo_matrix((weights, (rows, cols)), shape=(n, n)).tocsr()
    degree = ...  # flatten row sums to a one-dimensional array
    L_sparse = ...  # sparse.diags constructs D without a dense allocation
    return W_sparse, L_sparse.tocsr()

def csr_bytes(matrix):
    return matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes


In [ ]:
rows, cols, weights = affinity_edges(flat, connectivity=4)
W_dense, L_dense = dense_laplacian(6, rows, cols, weights)
W_sparse, L_sparse = sparse_laplacian(6, rows, cols, weights)
expected_L = np.array([[ 2,-1, 0,-1, 0, 0],
                       [-1, 3,-1, 0,-1, 0],
                       [ 0,-1, 2, 0, 0,-1],
                       [-1, 0, 0, 2,-1, 0],
                       [ 0,-1, 0,-1, 3,-1],
                       [ 0, 0,-1, 0,-1, 2]], dtype=float)
assert np.allclose(L_dense, expected_L)
assert np.allclose(L_sparse.toarray(), expected_L)
assert np.allclose(L_sparse @ np.ones(6), 0)
print("Laplacian checks passed. This tiny matrix is safe to print:")
display(pd.DataFrame(L_dense))

rows, cols, weights = fast_edges(image, 80.0, 4, 1e-6)
W_image, L_image = sparse_laplacian(H * W, rows, cols, weights)
print(f"Image Laplacian: {L_image.shape}, {L_image.nnz:,} stored entries")
print(f"CSR arrays: {csr_bytes(L_image)/2**20:.3f} MiB; hypothetical dense array: {(H*W)**2*8/2**20:.1f} MiB")
fig, ax = plt.subplots(figsize=(5, 5))
ax.spy(L_sparse, markersize=14)
ax.set_title("Nonzero pattern: the tiny 2-by-3 grid")
plt.show()


In [ ]:
storage = []
for side in [16, 32, 64, 128, 256]:
    n = side * side
    edges = 2 * side * (side - 1)  # undirected 4-neighbor edges
    nnz = n + 2 * edges
    storage.append({"image": f"{side} x {side}", "pixels": n, "L nonzeros": nnz,
                    "dense MiB": 8 * n * n / 2**20,
                    "CSR MiB (32-bit indices)": (8 * nnz + 4 * nnz + 4 * (n + 1)) / 2**20})
display(pd.DataFrame(storage))
print("These are matrix-storage estimates, excluding copies and solver factors/workspace.")


Explain one row of the printed Laplacian in terms of its pixel's neighbors. Why does it sum to zero? If both image dimensions double, how do the pixel count, dense storage, and grid sparse storage grow? Sparse factorization can introduce extra nonzeros (**fill-in**); why is the matrix-storage table not a peak-memory estimate for solving?


**Your response:** _Replace this text._


## 5. Mark foreground and background seeds

The supplied helpers below provide:

| Helper | Contract |
|---|---|
| `pick_seeds(image, seeds, names)` | Opens a Colab canvas with a class selector, Undo, Clear, Reset, and Save. Returns `[row, column, class]` triples. |
| `validate_seeds(seeds, shape, classes)` | Rejects out-of-bounds coordinates, conflicting labels, and missing classes; removes identical duplicate seeds. |
| `show_segmentation(image, probabilities, seeds, names, title)` | Shows seeds, probability maps with fixed 0–1 scales, and a color overlay with class labels. |

**Label convention:** class **1** is background, class **2** is foreground. The seed-label image uses **0** only to mean *unseeded*. A class label and a probability are different objects: a background seed has label 1 but foreground probability 0.

Run the helper cell as supplied. You may customize its appearance with AI. Preserve coordinate order, resizing behavior, class labels, and returned data types. An example request: “Modify this Matplotlib segmentation helper to use a different accessible palette and larger labels. Keep its inputs and outputs unchanged, preserve the 0–1 probability scale, and show me how to check that the mask did not change.”


In [ ]:
# @title Supplied seed and visualization helpers (expand to inspect or customize)
"""Source inserted into the assignment's supplied-helper cells."""

import base64
import io
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from PIL import Image

CLASS_COLORS = ["#2878b5", "#f28e2b", "#8b5fbf", "#2ca58d", "#d45087"]


def validate_seeds(seeds, shape, classes=(1, 2)):
    """Return unique integer [row, column, class] triples; reject conflicts."""
    raw = np.asarray(seeds)
    if raw.ndim != 2 or raw.shape[1] != 3 or len(raw) == 0:
        raise ValueError("Seeds must be a nonempty list of [row, column, class].")
    if not np.issubdtype(raw.dtype, np.number) or not np.all(np.isfinite(raw)):
        raise ValueError("Seed coordinates and labels must be finite numbers.")
    if not np.all(raw == np.floor(raw)):
        raise ValueError("Seed coordinates and labels must be integers.")
    a = raw.astype(np.int64)
    h, w = shape[:2]
    if np.any(a[:, 0] < 0) or np.any(a[:, 0] >= h) or np.any(a[:, 1] < 0) or np.any(a[:, 1] >= w):
        raise ValueError("A seed is outside this image. Check the image size and row/column order.")
    if not set(a[:, 2]).issubset(set(classes)):
        raise ValueError(f"Use only class labels {list(classes)}.")
    unique = {}
    for r, c, label in a:
        if (r, c) in unique and unique[r, c] != label:
            raise ValueError("One pixel cannot belong to two seed classes.")
        unique[r, c] = label
    if set(unique.values()) != set(classes):
        raise ValueError("Place at least one seed for every class.")
    return np.array([[r, c, label] for (r, c), label in sorted(unique.items())], dtype=np.int64)


def draw_seeds(ax, seeds, names):
    for label, name in enumerate(names, 1):
        a = np.asarray(seeds)
        a = a[a[:, 2] == label]
        if len(a):
            ax.scatter(a[:, 1], a[:, 0], s=65, c=CLASS_COLORS[label - 1],
                       edgecolors="white", linewidths=1.4, label=f"{label}: {name}")
            for r, c, _ in a:
                ax.annotate(str(label), (c, r), xytext=(4, 5), textcoords="offset points",
                            color="white", fontsize=8, weight="bold",
                            bbox=dict(facecolor="black", alpha=.5, pad=.5, edgecolor="none"))


def show_segmentation(image, probabilities, seeds, names=("background", "foreground"), title=""):
    """Display saved seeds, probability maps, and a labeled color overlay.

    Binary input: H-by-W probability of class 2; threshold >= .5.
    Multiclass input: H-by-W-by-k probabilities; argmax (first class wins ties).
    Values are displayed as computed; this function never clips a solver's output.
    """
    p = np.asarray(probabilities)
    k = len(names)
    seeds = validate_seeds(seeds, image.shape, tuple(range(1, k + 1)))
    if p.ndim == 2:
        assert k == 2 and p.shape == image.shape[:2]
        maps = [p]
        captions = ["Probability of class 2"]
        labels = (p >= .5).astype(int)
    else:
        assert p.shape == (*image.shape[:2], k)
        maps = [p[..., j] for j in range(k)]
        captions = [f"Probability of class {j + 1}" for j in range(k)]
        labels = p.argmax(axis=2)
    fig, axes = plt.subplots(1, len(maps) + 2, figsize=(4.3 * (len(maps) + 2), 4.5))
    axes[0].imshow(image)
    draw_seeds(axes[0], seeds, names)
    axes[0].set_title("Image and saved seeds")
    for ax, values, caption in zip(axes[1:-1], maps, captions):
        im = ax.imshow(values, cmap="viridis", vmin=0, vmax=1)
        ax.set_title(caption)
        fig.colorbar(im, ax=ax, fraction=.046, pad=.04)
    axes[-1].imshow(image)
    axes[-1].imshow(labels, cmap=ListedColormap(CLASS_COLORS[:k]), vmin=-.5, vmax=k - .5, alpha=.42)
    axes[-1].set_title("Segmentation overlay")
    for ax in axes:
        ax.set_axis_off()
    handles = [Patch(facecolor=CLASS_COLORS[j], label=f"{j + 1}: {name}") for j, name in enumerate(names)]
    fig.legend(handles=handles, loc="lower center", ncol=k, frameon=False)
    fig.suptitle(title, fontsize=14)
    fig.tight_layout(rect=(0, .08, 1, .92))
    plt.show()
    return labels + 1


def picker_script(image, seeds, names):
    """Build the self-contained JavaScript used by the Colab pixel picker."""
    if not 2 <= len(names) <= len(CLASS_COLORS):
        raise ValueError("The picker supports two to five classes.")
    buffer = io.BytesIO()
    Image.fromarray(np.round(np.clip(image, 0, 1) * 255).astype(np.uint8)).save(buffer, format="PNG")
    payload = dict(image="data:image/png;base64," + base64.b64encode(buffer.getvalue()).decode(),
                   seeds=np.asarray(seeds, dtype=int).tolist() if len(seeds) else [],
                   names=list(names), colors=CLASS_COLORS[:len(names)])
    return r"""(async () => {
      const config = __PAYLOAD__;
      const root = document.createElement('div');
      root.style.cssText = 'font:15px system-ui;max-width:760px;padding:14px;border:1px solid #ccd4df;border-radius:12px;background:#fff;color:#172436';
      const heading = document.createElement('h3');
      heading.textContent = 'Choose seed pixels'; root.appendChild(heading);
      const help = document.createElement('p');
      help.textContent = 'Select a class, then click its pixels. Clicking an existing seed changes its class. Save returns [row, column, class] triples.';
      root.appendChild(help);
      const toolbar = document.createElement('div');
      toolbar.style.cssText = 'display:flex;gap:8px;flex-wrap:wrap;margin:10px 0';
      const select = document.createElement('select'); select.setAttribute('aria-label', 'Seed class');
      config.names.forEach((name, i) => { const option = document.createElement('option'); option.value = i + 1; option.textContent = `${i + 1}: ${name}`; select.appendChild(option); });
      toolbar.appendChild(select);
      const button = (name) => { const b = document.createElement('button'); b.textContent=name; b.type='button'; b.style.cssText='padding:7px 12px;cursor:pointer'; toolbar.appendChild(b); return b; };
      const undo = button('Undo'), clear = button('Clear'), reset = button('Reset'), done = button('Save seeds');
      root.appendChild(toolbar);
      const canvas = document.createElement('canvas');
      canvas.style.cssText = 'width:100%;max-width:640px;height:auto;cursor:crosshair;display:block;border:1px solid #aaa';
      canvas.setAttribute('aria-label','Image: click to place seeds');
      root.appendChild(canvas);
      const status = document.createElement('p'); status.setAttribute('role','status'); root.appendChild(status);
      document.body.appendChild(root);
      const picture = new Image();
      await new Promise((resolve,reject) => { picture.onload=resolve; picture.onerror=reject; picture.src=config.image; });
      canvas.width=picture.naturalWidth; canvas.height=picture.naturalHeight;
      const context=canvas.getContext('2d');
      let points=config.seeds.map(p=>p.slice()), history=[];
      const remember = () => history.push(points.map(p=>p.slice()));
      const redraw = () => {
        context.drawImage(picture,0,0);
        const radius=Math.max(2.2,canvas.width/90);
        for(const [r,c,label] of points) {
          context.beginPath(); context.arc(c+.5,r+.5,radius,0,2*Math.PI);
          context.fillStyle=config.colors[label-1]; context.fill();
          context.strokeStyle='white'; context.lineWidth=Math.max(.7,canvas.width/400); context.stroke();
          context.font=`bold ${Math.max(6,canvas.width/40)}px sans-serif`;
          context.fillStyle='white'; context.fillText(String(label),c+radius,r-radius);
        }
        status.textContent=config.names.map((name,i)=>`${i+1}: ${points.filter(p=>p[2]===i+1).length} seeds`).join(' | ');
      };
      canvas.onclick=(event)=>{
        const box=canvas.getBoundingClientRect();
        const c=Math.min(canvas.width-1,Math.max(0,Math.floor((event.clientX-box.left)*canvas.width/box.width)));
        const r=Math.min(canvas.height-1,Math.max(0,Math.floor((event.clientY-box.top)*canvas.height/box.height)));
        remember(); points=points.filter(p=>p[0]!==r || p[1]!==c);
        points.push([r,c,Number(select.value)]); redraw();
      };
      undo.onclick=()=>{ if(history.length) points=history.pop(); redraw(); };
      clear.onclick=()=>{ remember(); points=[]; redraw(); };
      reset.onclick=()=>{ remember(); points=config.seeds.map(p=>p.slice()); redraw(); };
      redraw();
      return await new Promise(resolve=>{
        done.onclick=()=>{
          if(config.names.some((_,i)=>!points.some(p=>p[2]===i+1))) {
            status.textContent='Add at least one seed for every class before saving.'; return;
          }
          points.sort((a,b)=>a[0]-b[0] || a[1]-b[1]);
          status.textContent=`Saved ${points.length} seeds. Copy the configuration assignment printed below into your notebook, then turn the corresponding edit switch off.`;
          canvas.onclick=null;
          for(const control of [select,undo,clear,reset,done]) control.disabled=true;
          resolve(points);
        };
      });
    })()""".replace("__PAYLOAD__", json.dumps(payload))


def pick_seeds(image, seeds=(), names=("background", "foreground")):
    """Open the Colab picker and return integer [row, column, class] triples."""
    try:
        from google.colab import output
    except ImportError as exc:
        raise RuntimeError("Open this notebook in Colab to click seeds, or edit SAVED_CASE manually.") from exc
    picked = output.eval_js(picker_script(image, seeds, names), timeout_sec=600)
    return validate_seeds(picked, image.shape, tuple(range(1, len(names) + 1)))


### Save clicks as data

The configuration below starts with demonstration seeds for image 2 at 128 pixels on its longer side. First run it unchanged. To choose your own seeds:

1. Set `EDIT_SEEDS = True` and run the configuration and picker cells.
2. Select a class and click several pixels inside that region; use both classes, then press **Save seeds**.
3. Copy the **entire `SAVED_CASE = ...` assignment printed by the picker** into the configuration cell, replacing its previous value.
4. Set `EDIT_SEEDS = False` and rerun the configuration and picker cells. The notebook now reproduces your clicks without waiting for interaction.

If you change the image or its resolution, choose new seeds. The saved image number and shape prevent silently reusing coordinates on a different image. Colab runtime files alone are temporary; the pasted configuration travels with your notebook. If the canvas is unavailable, enter triples manually using the coordinate plot above.


In [ ]:
EDIT_SEEDS = False
SAVED_CASE = {"image_number": 2, "shape": [128, 128],
              "seeds": [[5, 5, 1], [5, 122, 1], [122, 5, 1], [122, 122, 1],
                        [48, 52, 2], [65, 74, 2], [85, 56, 2]]}
CLASS_NAMES = ("background", "foreground")


In [ ]:
saved_matches = (SAVED_CASE["image_number"] == IMAGE_NUMBER and
                 SAVED_CASE["shape"] == list(image.shape[:2]))
if EDIT_SEEDS:
    starting_seeds = SAVED_CASE["seeds"] if saved_matches else []
    seeds = pick_seeds(image, starting_seeds, CLASS_NAMES)
    captured_case = {"image_number": IMAGE_NUMBER, "shape": list(image.shape[:2]),
                     "seeds": seeds.tolist()}
    print("Copy the following assignment into the configuration cell, then turn EDIT_SEEDS off:")
    print("SAVED_CASE = " + repr(captured_case))
else:
    if not saved_matches:
        raise ValueError("Saved seeds belong to a different image or size. Use the picker or update SAVED_CASE.")
    seeds = validate_seeds(SAVED_CASE["seeds"], image.shape)
    print("Replaying", len(seeds), "saved seeds.")

seed_labels = np.zeros((H, W), dtype=np.int64)
seed_labels[seeds[:, 0], seeds[:, 1]] = seeds[:, 2]
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image); draw_seeds(ax, seeds, CLASS_NAMES)
ax.legend(loc="upper left", bbox_to_anchor=(1, 1)); ax.set_title("Your fixed boundary values")
ax.set(xlabel="column", ylabel="row"); plt.tight_layout(); plt.show()


Describe the object you want to separate. Why did you put your seeds at these locations? What ambiguity do you expect near its boundary? If you modified a helper with AI, record the actual prompt, the change you kept, and a check of its behavior. Otherwise state that you used the supplied helper unchanged.


**Your response:** _Replace this text._


## 6. Turn fixed seeds into a linear system

Let $x_i$ be the probability that a random walker starting at pixel $i$ reaches a foreground seed before a background seed. At each step it chooses neighbor $j$ with probability $w_{ij}/d_i$. At an unseeded pixel,
$$
x_i=\frac{1}{d_i}\sum_j w_{ij}x_j
\quad\Longrightarrow\quad
d_ix_i-\sum_j w_{ij}x_j=0.
$$
No random-walk simulation is needed: we can solve for these probabilities directly.

Split the vertices into seeds $S$ and unknowns $U$. The seed values $y_S$ are 0 for background and 1 for foreground. The unknown rows of $Lx=0$ give
$$
L_{UU}x_U+L_{US}y_S=0,
\qquad \boxed{A=L_{UU},\quad b=-L_{US}y_S,\quad Ax_U=b.}
$$
**Keep the original degrees.** Slice the full graph Laplacian; do not build a new Laplacian on the unknown pixels alone. Edges from unknown pixels to seeds still contribute to the diagonal.

The full $L$ is singular. The reduced $A$ is symmetric positive definite when every connected component contains a seed. Our positive grid weights make the graph connected; both classes must have a seed for this binary task.

### A four-pixel example

Take a constant-color path with vertices `0 — 1 — 2 — 3`. Vertex 0 is foreground and vertex 3 is background. The equations are $2x_1-x_2=1$ and $-x_1+2x_2=0$. Work out the two probabilities before running the check.


Write $A$ and $b$ for the four-pixel example, then solve it by hand. Explain the minus sign in $b=-L_{US}y_S$. Why would dropping edges to the seeds from the diagonal change the equations?


**Your response:** _Replace this text._


### Your implementations 3 and 4: eliminate seeds and solve

Complete `seeded_system` and the two solver calls in `solve_binary`. The indexing `L[U][:, S]` selects all combinations of unknown rows and seed columns. Using `L[U, S]` would request paired indexing instead.


In [ ]:
def seeded_system(L, seeds, shape):
    seeds = validate_seeds(seeds, shape)
    h, w = shape[:2]
    n = h * w
    assert L.shape == (n, n)
    S = seeds[:, 0] * w + seeds[:, 1]
    yS = (seeds[:, 2] == 2).astype(np.float64)
    unknown = np.ones(n, dtype=bool)
    unknown[S] = False
    U = np.flatnonzero(unknown)
    if len(U) == 0:
        raise ValueError("Leave at least one unseeded pixel for this experiment.")
    A = ...  # slice unknown rows and unknown columns, then convert to CSR
    b = ...  # use unknown rows, seed columns, and fixed seed probabilities
    return A, np.asarray(b).ravel(), U, S, yS

def solve_binary(L, seeds, shape, method="sparse"):
    A, b, U, S, yS = seeded_system(L, seeds, shape)
    if method == "dense":
        if L.shape[0] > 1600:
            raise ValueError("Dense safety limit: use at most 1,600 pixels.")
        xU = ...  # numpy.linalg.solve; do not form an inverse
    elif method == "sparse":
        xU = ...  # scipy.sparse.linalg.spsolve
    else:
        raise ValueError("method must be 'dense' or 'sparse'")
    x = np.empty(L.shape[0], dtype=np.float64)
    x[S], x[U] = yS, xU
    relative_residual = np.linalg.norm(A @ xU - b) / max(np.linalg.norm(b), 1e-15)
    assert np.all(np.isfinite(x))
    assert x.min() >= -1e-7 and x.max() <= 1 + 1e-7, "Check the equations; do not hide errors by clipping."
    assert relative_residual < 1e-7, "Large residual: inspect the system and numerical conditioning."
    return x.reshape(shape[:2]), relative_residual


In [ ]:
path_image = np.zeros((1, 4, 3), dtype=np.float64)
path_edges = affinity_edges(path_image)
_, path_L = sparse_laplacian(4, *path_edges)
path_seeds = [[0, 0, 2], [0, 3, 1]]
A_test, b_test, *_ = seeded_system(path_L, path_seeds, (1, 4))
assert np.allclose(A_test.toarray(), [[2, -1], [-1, 2]])
assert np.allclose(b_test, [1, 0])
for method in ("dense", "sparse"):
    p_test, residual = solve_binary(path_L, path_seeds, (1, 4), method)
    assert np.allclose(p_test, [[1, 2/3, 1/3, 0]])
print("Seed elimination and both solvers passed the path check.")
print("Foreground probabilities:", p_test)


In [ ]:
BETA = 80.0
CONNECTIVITY = 4
rows, cols, weights = fast_edges(image, BETA, CONNECTIVITY, 1e-6)
_, L_image = sparse_laplacian(H * W, rows, cols, weights)
probability, residual = solve_binary(L_image, seeds, image.shape)
assert np.array_equal(probability[seeds[:, 0], seeds[:, 1]], (seeds[:, 2] == 2).astype(float))
segmentation = show_segmentation(image, probability, seeds, CLASS_NAMES,
                                 f"{CONNECTIVITY}-neighbors | beta={BETA:g} | {H} x {W}")
print(f"Relative residual: {residual:.2e}; probability range: [{probability.min():.6f}, {probability.max():.6f}]")
print("Binary decision: class 2 if foreground probability >= 0.5, otherwise class 1.")


Identify a region with probabilities near 0.5 and a region near 0 or 1. Does the boundary match your intended object? A small residual says that the equations were solved accurately; does it establish that the image was segmented correctly? Explain the difference.


**Your response:** _Replace this text._


## 7. Compare dense and sparse solutions fairly

For each image size, both solvers receive the **same reduced system and seeds**. Seed coordinates are scaled to each resolution; if two seeds collide, that size is skipped explicitly. Comparisons across resolutions are different discretizations of the same image, not identical systems.

Record graph construction, sparse-to-dense conversion, and solve time separately. The solver timers include factorization and solution, exclude graph construction and conversion, and use medians of three fresh calls. The dense solver runs with one BLAS thread so an unspecified thread pool does not dominate the comparison. Dense and sparse solvers use different algorithms; their speed ratio reflects both storage and solver implementation.

The sweep uses longest sides 16, 24, and 32. Larger resolutions use sparse solving only. A dense direct solve generally costs $O(n^3)$ arithmetic and $O(n^2)$ storage for $n$ unknowns. Sparse storage for this grid is $O(n)$, but sparse direct factorization can fill in and is **not** guaranteed linear-time or linear-memory.


In [ ]:
def resize_rgb(image, max_side):
    pil = Image.fromarray(np.round(image * 255).astype(np.uint8))
    pil.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
    return np.ascontiguousarray(np.asarray(pil, dtype=np.float64) / 255.0)

def scale_seeds(seeds, old_shape, new_shape):
    old_h, old_w = old_shape[:2]
    new_h, new_w = new_shape[:2]
    scaled = np.asarray(seeds).copy()
    scaled[:, 0] = np.rint(scaled[:, 0] * (new_h - 1) / max(old_h - 1, 1)).astype(int)
    scaled[:, 1] = np.rint(scaled[:, 1] * (new_w - 1) / max(old_w - 1, 1)).astype(int)
    if len(set(map(tuple, scaled[:, :2]))) != len(scaled):
        raise ValueError("Two seeds collide at this resolution; use a larger benchmark size or more separated seeds.")
    return validate_seeds(scaled, new_shape)

def make_graph(small_image, beta, connectivity):
    return sparse_laplacian(np.prod(small_image.shape[:2]),
                            *fast_edges(small_image, beta, connectivity, 1e-6))[1]

benchmark_rows = []
with threadpool_limits(limits=1, user_api="blas"):
    for side in [16, 24, 32]:
        small = resize_rgb(image, side)
        try:
            small_seeds = scale_seeds(seeds, image.shape, small.shape)
        except ValueError as error:
            print(f"Skipping side={side}: {error}")
            continue
        graph_s, L_small = median_seconds(lambda: make_graph(small, BETA, CONNECTIVITY))
        A, b, *_ = seeded_system(L_small, small_seeds, small.shape)
        conversion_s, A_dense = median_seconds(A.toarray)
        # Untimed warm-up of the numerical libraries, separate from measured solves.
        np.linalg.solve(A_dense, b); spsolve(A, b)
        dense_s, xd = median_seconds(lambda: np.linalg.solve(A_dense, b))
        sparse_s, xs = median_seconds(lambda: spsolve(A, b))
        difference = float(np.max(np.abs(xd - xs)))
        assert np.allclose(xd, xs, rtol=1e-7, atol=1e-8)
        rd = np.linalg.norm(A_dense @ xd - b) / max(np.linalg.norm(b), 1e-15)
        rs = np.linalg.norm(A @ xs - b) / max(np.linalg.norm(b), 1e-15)
        assert max(rd, rs) < 1e-7
        benchmark_rows.append({"longest side": side, "pixels": int(np.prod(small.shape[:2])),
            "unknowns": A.shape[0], "graph ms": 1000 * graph_s, "to-dense ms": 1000 * conversion_s,
            "NumPy solve ms": 1000 * dense_s, "SciPy sparse solve ms": 1000 * sparse_s,
            "dense A MiB": A_dense.nbytes / 2**20, "CSR A MiB": csr_bytes(A) / 2**20,
            "max difference": difference, "dense residual": rd, "sparse residual": rs})
        del A_dense

benchmark = pd.DataFrame(benchmark_rows)
assert len(benchmark) > 0, "All sizes had seed collisions; select well-separated benchmark seeds."
display(benchmark)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for column in ["NumPy solve ms", "SciPy sparse solve ms"]:
    axes[0].plot(benchmark["unknowns"], benchmark[column], "o-", label=column)
for column in ["dense A MiB", "CSR A MiB"]:
    axes[1].plot(benchmark["unknowns"], benchmark[column], "o-", label=column)
axes[0].set(xlabel="unknowns", ylabel="median solve time (ms)", yscale="log", title="Same equations, two solvers")
axes[1].set(xlabel="unknowns", ylabel="stored matrix MiB", yscale="log", title="Stored reduced matrix")
for ax in axes: ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

start = time.perf_counter()
large_probability, large_residual = solve_binary(L_image, seeds, image.shape)
print(f"Sparse-only main image ({H*W:,} pixels): {time.perf_counter()-start:.3f} s, including system slicing.")


Use your measured values to discuss when sparse storage and solving became useful. Is sparse solving necessarily faster on the smallest problem? Explain why dense/sparse agreement and a small residual are both useful checks. If the longest image side doubles, how would the general dense operation count scale? Distinguish a measured trend from an asymptotic claim, and state what your timers include.


**Your response:** _Replace this text._


## 8. Investigate the model

Change one thing at a time. The first cell compares two beta values with the same seeds and graph topology, then changes connectivity with beta held fixed. These are starting experiments; adjust the values if your image needs them.

Next, compare a small seed set (one seed per class) with your full seed set while keeping beta and connectivity fixed. If these coincide, add more seeds and repeat. Keep static figures and write conclusions that refer to specific regions.


In [ ]:
experiment_results = []
for beta, connectivity in [(20.0, 4), (80.0, 4), (80.0, 8)]:
    L = make_graph(image, beta, connectivity)
    p, residual = solve_binary(L, seeds, image.shape)
    labels = show_segmentation(image, p, seeds, CLASS_NAMES,
                               f"Fixed seeds | beta={beta:g}, {connectivity}-neighbors")
    experiment_results.append({"beta": beta, "neighbors": connectivity,
                               "foreground fraction": np.mean(labels == 2), "relative residual": residual})
display(pd.DataFrame(experiment_results))

few_seeds = np.array([seeds[seeds[:, 2] == label][0] for label in (1, 2)])
L_fixed = make_graph(image, BETA, CONNECTIVITY)
for name, chosen in [("One seed per class", few_seeds), ("All saved seeds", seeds)]:
    p, _ = solve_binary(L_fixed, chosen, image.shape)
    show_segmentation(image, p, chosen, CLASS_NAMES, f"{name} | fixed beta and connectivity")


Describe what changed in each controlled comparison. Which change had the largest effect, and where? Explain one result using the affinity weights or seed constraints. Include one failure or ambiguous boundary; do not report only the best-looking result. Foreground fraction measures output size, not segmentation accuracy.


**Your response:** _Replace this text._


### Transfer your method to another image

Choose a different image from the catalog, state its intended foreground, and select at least two seeds per class. Image 22, the Sun, is a starting example. Its glow makes the intended boundary debatable, and the starter seeds may capture only part of the disk. Use the same save-and-replay procedure, this time copying the printed `SECOND_CASE` assignment into this cell. You may adjust beta or add seeds after inspecting the result; document the adjustment.


In [ ]:
SECOND_NUMBER = 22
SECOND_SIDE = 128
SECOND_BETA = 80.0
EDIT_SECOND_SEEDS = False
SECOND_CASE = {"image_number": 22, "shape": [128, 128],
               "seeds": [[8, 8, 1], [8, 119, 1], [119, 8, 1], [119, 119, 1],
                         [46, 45, 2], [64, 64, 2], [78, 87, 2]]}


In [ ]:
assert SECOND_NUMBER != IMAGE_NUMBER, "Choose a different second image."
second_image, second_info = load_image(SECOND_NUMBER, SECOND_SIDE)
print(f"{second_info['title']} — {second_info['creator']}")
print("Source:", second_info["source"])
second_matches = (SECOND_CASE["image_number"] == SECOND_NUMBER and
                  SECOND_CASE["shape"] == list(second_image.shape[:2]))
if EDIT_SECOND_SEEDS:
    second_seeds = pick_seeds(second_image, SECOND_CASE["seeds"] if second_matches else [], CLASS_NAMES)
    print("SECOND_CASE = " + repr({"image_number": SECOND_NUMBER,
          "shape": list(second_image.shape[:2]), "seeds": second_seeds.tolist()}))
else:
    if not second_matches:
        raise ValueError("SECOND_CASE does not match this image and resolution. Choose and save new seeds.")
    second_seeds = validate_seeds(SECOND_CASE["seeds"], second_image.shape)
second_L = make_graph(second_image, SECOND_BETA, CONNECTIVITY)
second_p, second_residual = solve_binary(second_L, second_seeds, second_image.shape)
show_segmentation(second_image, second_p, second_seeds, CLASS_NAMES,
                   f"Second image | beta={SECOND_BETA:g} | {CONNECTIVITY}-neighbors")
print("Second-image relative residual:", second_residual)


What is foreground in the second image? Describe how you chose the seeds and whether the parameter settings transferred successfully. Identify a limitation of RGB affinity on this image. State your intended boundary even if the segmentation misses it.


**Your response:** _Replace this text._


## 9. Bonus: more than two classes

Leo Grady's *Random Walks for Image Segmentation* (IEEE TPAMI, 2006, DOI [10.1109/TPAMI.2006.233](https://doi.org/10.1109/TPAMI.2006.233)) develops the multiclass method. The binary method above is already a special case. Consult the algorithm and Dirichlet-problem discussion in the [paper](https://vision.cse.psu.edu/people/chenpingY/paper/grady2006random.pdf).

For $k$ classes, make the seed matrix $Y_S$ one-hot: its entry $(s,c)$ is 1 if seed $s$ has class $c$, otherwise 0. Solve
$$
L_{UU}P_U=-L_{US}Y_S.
$$
Each column gives the probability of first reaching a seed of that class. Assign each pixel its largest-probability class. With all components seeded, each probability row sums to one, so only $k-1$ systems are mathematically independent. You may solve all $k$ for clarity.

**Bonus work:** complete the function below, reuse one sparse factorization for all right-hand sides, verify probability bounds and row sums, and try at least three classes with multiple seeds per class on an image from the repository. Compare against the two-class task and explain which regions you separated.

Start by switching on the bonus and passing the synthetic three-region example. Then use `pick_seeds(..., names=(...))` to collect your three-class image seeds, paste them into a saved configuration, and add the code and explanation in the cells below. Keep the bonus switch false if you are not attempting this part.


In [ ]:
RUN_BONUS = False

def solve_multiclass(L, seeds, shape, k):
    seeds = validate_seeds(seeds, shape, tuple(range(1, k + 1)))
    n = L.shape[0]
    S = seeds[:, 0] * shape[1] + seeds[:, 1]
    U = np.setdiff1d(np.arange(n), S)
    YS = ...  # one-hot rows in the same order as S; shape (len(S), k)
    A = L[U][:, U].tocsc()
    B = ...  # all k right-hand sides; shape (len(U), k)
    factor = ...  # scipy.sparse.linalg.splu(A), once
    PU = ...  # use the factor's solve method on the whole B matrix
    P = np.empty((n, k), dtype=np.float64)
    P[S], P[U] = YS, PU
    assert np.all(np.isfinite(P))
    assert P.min() >= -1e-7 and P.max() <= 1 + 1e-7
    assert np.allclose(P.sum(axis=1), 1, atol=1e-7)
    assert np.array_equal(P[S], YS)
    assert np.linalg.norm(A @ PU - B) / max(np.linalg.norm(B), 1e-15) < 1e-7
    return P.reshape(*shape[:2], k)

if RUN_BONUS:
    toy = np.empty((24, 36, 3), dtype=np.float64)
    toy[:, :12] = [0.8, 0.2, 0.15]
    toy[:, 12:24] = [0.2, 0.65, 0.25]
    toy[:, 24:] = [0.15, 0.3, 0.85]
    toy_seeds = [[5, 4, 1], [18, 7, 1], [5, 16, 2], [18, 20, 2], [5, 28, 3], [18, 31, 3]]
    toy_L = make_graph(toy, 20.0, 4)
    toy_P = solve_multiclass(toy_L, toy_seeds, toy.shape, 3)
    show_segmentation(toy, toy_P, toy_seeds, ("left", "middle", "right"), "Three classes, one factorization")
    print("Multiclass checks passed.")
else:
    print("Bonus skipped. Set RUN_BONUS = True after completing solve_multiclass.")


In [ ]:
# Optional: add your repository-image experiment here.
# Save the image number, resolution, class names, and seed triples in this cell.
# Guard interactive clicking with a switch that is False in your final submission.


**If attempting the bonus:** explain the meaning of a column of `P`, why its rows sum to one, and what computation is reused across classes. Include a repository-image result with at least three classes and multiple seeds per class, and cite the Grady paper. Otherwise write “Bonus not attempted.”


**Your response:** _Replace this text._


## 10. Reflection and submission

Write **250–400 words** reflecting on your process. Refer to your own outputs and measurements.

- Explain the path from RGB values to affinities, a Laplacian, and a seeded system. Identify one idea that became clearer through implementation.
- Describe a bug, incorrect assumption, or difficult step, and the evidence you used to resolve it.
- Explain one runtime or memory result, distinguishing compilation, storage, and solving.
- Discuss a segmentation that did not match your intention and what you would try next.
- State how you used AI, if at all, and how you checked its contribution. Include an actual helper-customization prompt if you used one.

**Your reflection:** _Replace this text._

### Before submitting

- Complete all four implementation exercises and written responses.
- Keep the main and second-image seed configurations in the notebook; turn both edit switches off.
- Keep the probability maps, segmentation overlays, and runtime/memory comparisons visible.
- Include the controlled comparisons and a second image with your intended foreground explained.
- Use the bonus switch only if your bonus code works; skip it explicitly otherwise.
- Restart the runtime and run all cells from top to bottom. Confirm there are no errors or stale outputs.
- Save and download the executed `.ipynb`. Your saved seed configurations must reproduce the figures without more clicks.


In [ ]:
assert not EDIT_SEEDS and not EDIT_SECOND_SEEDS, "Turn off interactive editing before your final Run all."
assert SAVED_CASE["image_number"] == IMAGE_NUMBER and SAVED_CASE["shape"] == list(image.shape[:2])
assert SECOND_CASE["image_number"] == SECOND_NUMBER and SECOND_CASE["shape"] == list(second_image.shape[:2])
assert np.array_equal(validate_seeds(SAVED_CASE["seeds"], image.shape), seeds)
assert np.array_equal(validate_seeds(SECOND_CASE["seeds"], second_image.shape), second_seeds)
assert np.isfinite(benchmark.select_dtypes(include="number").to_numpy()).all()
assert len(experiment_results) >= 3
print("Computational checks complete. Review every written response and your reflection before downloading.")


## References

- Leo Grady, “Random Walks for Image Segmentation,” *IEEE Transactions on Pattern Analysis and Machine Intelligence* 28(11), 1768–1783, 2006. [DOI](https://doi.org/10.1109/TPAMI.2006.233).
- [Numba: a five-minute guide](https://numba.readthedocs.io/en/stable/user/5minguide.html), especially compilation and timing.
- [NumPy `linalg.solve`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html).
- [SciPy sparse matrices](https://docs.scipy.org/doc/scipy/reference/sparse.html), [`spsolve`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.linalg.spsolve.html), and [`splu`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.linalg.splu.html).
- [Colab JavaScript evaluation interface](https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_js.py), used by the supplied seed picker.
- [Course image catalog and credits](https://ikoutis.github.io/cs301-labs/unit-2/images/SOURCES.md).
